# 🎙️ Fish Audio S2 Pro — Streaming Test
**Repo:** [ParthThakkar08/audio.cpp](https://github.com/ParthThakkar08/audio.cpp)  
**Goal:** Build → Load model → Start streaming server → Verify chunked PCM output → Benchmark TTFA

> ⚙️ Runtime → Change runtime type → **GPU → A100**

In [ ]:
# 1. Verify A100
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(out)
assert 'A100' in out, '❌ Switch runtime to A100 GPU first!'
print('✅ A100 confirmed')

In [ ]:
# 2. Install build deps
!apt-get update -qq && apt-get install -y -qq cmake ninja-build git
!pip install -q huggingface_hub

In [ ]:
# 3. Clone repo
import os
REPO = '/content/audio.cpp'
if not os.path.exists(REPO):
    !git clone --recurse-submodules https://github.com/ParthThakkar08/audio.cpp.git {REPO}
else:
    !git -C {REPO} pull --recurse-submodules
!git -C {REPO} log --oneline -3

In [ ]:
# 4. Build audiocpp_server (CUDA + fast-math)
import subprocess, os
BUILD = f'{REPO}/build'
os.makedirs(BUILD, exist_ok=True)

r = subprocess.run(
    ['cmake', '..', '-GNinja',
     '-DCMAKE_BUILD_TYPE=Release',
     '-DAUDIO_CPP_CUDA=ON',
     '-DCMAKE_CUDA_FLAGS=--use_fast_math',
     '-DCMAKE_CXX_FLAGS=-O3'],
    cwd=BUILD, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-3000:]); raise RuntimeError('CMake failed')

r = subprocess.run(['ninja', '-j8', 'audiocpp_server'],
                   cwd=BUILD, capture_output=True, text=True)
print(r.stdout[-1000:])
if r.returncode != 0:
    print(r.stderr[-3000:]); raise RuntimeError('Build failed')

!ls -lh {BUILD}/bin/audiocpp_server
print('✅ Build complete')

In [ ]:
# 5. Download Fish Audio S2 Pro Q8_0 GGUF (~6.3 GB)
from huggingface_hub import snapshot_download
MODEL_DIR = '/content/models'

snapshot_download(
    repo_id='audio-cpp/audio.cpp-gguf',
    allow_patterns=[
        'Fish-Audio-S2-Pro-GGUF/fish-audio-s2-pro-q8_0.gguf',
        'Fish-Audio-S2-Pro-GGUF/config.json',
        'Fish-Audio-S2-Pro-GGUF/tokenizer.json',
        'Fish-Audio-S2-Pro-GGUF/tokenizer_config.json',
    ],
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False,
)
!ls -lh {MODEL_DIR}/Fish-Audio-S2-Pro-GGUF/
print('✅ Model ready')

In [ ]:
# 6. Set up prakash_confident voice preset
# Upload your reference WAV here OR use the URL approach below.
import os

VOICE_NAME  = 'prakash_confident'
REFS_DIR    = f'{MODEL_DIR}/Fish-Audio-S2-Pro-GGUF/references/{VOICE_NAME}'
os.makedirs(REFS_DIR, exist_ok=True)

# ── Option A: upload from local machine ──────────────────────────────────────
# from google.colab import files
# up = files.upload()          # pick prakash_confident.wav
# import shutil
# shutil.copy(list(up)[0], f'{REFS_DIR}/{VOICE_NAME}.wav')

# ── Option B: download from URL ───────────────────────────────────────────────
# VOICE_URL = 'https://your-host.com/prakash_confident.wav'
# !wget -q -O {REFS_DIR}/{VOICE_NAME}.wav "{VOICE_URL}"

# Reference transcript (.lab file)
LAB = 'Namaste, main aapki kis tarah madad kar sakta hoon?'
open(f'{REFS_DIR}/{VOICE_NAME}.lab', 'w').write(LAB)

print(f'Reference dir: {REFS_DIR}')
!ls -lh {REFS_DIR}/

In [ ]:
# 7. Write server_config.json (streaming mode)
import json

CONFIG = '/content/server_config.json'
cfg = {
    'model_family': 'fish_audio',
    'mode': 'streaming',
    'package': 'fish_audio_s2_pro_q8_0',
    'model_dir': f'{MODEL_DIR}/Fish-Audio-S2-Pro-GGUF',
    'server': {'host': '127.0.0.1', 'port': 8020},
    'backend': {'device': 'cuda', 'threads': 4},
    'session_options': {'reference_cache_slots': 4}
}
json.dump(cfg, open(CONFIG, 'w'), indent=2)
print(json.dumps(cfg, indent=2))

In [ ]:
# 8. Start audiocpp_server
import subprocess, time, urllib.request, os

SERVER = f'{BUILD}/bin/audiocpp_server'
LOG    = '/content/server.log'

!pkill -f audiocpp_server 2>/dev/null; sleep 1

proc = subprocess.Popen(
    [SERVER, '--config', CONFIG],
    stdout=open(LOG, 'w'), stderr=subprocess.STDOUT,
    preexec_fn=os.setsid
)
print(f'Server PID: {proc.pid}')

for i in range(40):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8020/v1/models', timeout=2)
        print(f'✅ Server ready in {(i+1)*2}s')
        break
    except:
        print(f'  waiting... {(i+1)*2}s', end='\r')
else:
    !tail -40 {LOG}
    raise RuntimeError('Server failed to start')

In [ ]:
# 9. Cloudflare tunnel
import subprocess, time, re, os

CF = '/content/cloudflared'
if not os.path.exists(CF):
    !wget -q -O {CF} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x {CF}

CF_LOG = '/content/cf.log'
cf = subprocess.Popen([CF, 'tunnel', '--url', 'http://127.0.0.1:8020', '--no-autoupdate'],
                      stdout=open(CF_LOG, 'w'), stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(30):
    time.sleep(2)
    log = open(CF_LOG).read()
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', log)
    if m:
        PUBLIC_URL = m.group(0)
        break

if PUBLIC_URL:
    print(f'\n🌐 TTS STREAMING URL:\n{PUBLIC_URL}/v1/audio/speech')
else:
    !tail -20 {CF_LOG}
    raise RuntimeError('Tunnel failed')

In [ ]:
# 10. Streaming verification — confirms Transfer-Encoding: chunked with multiple chunks
import http.client, json, time

TEXT = (
    'Namaste! Aaj hum aapke account ke baare mein baat karenge. '
    'Kya aap mujhe apna registered mobile number bata sakte hain? '
    'Main aapki poori madad karne ke liye yahan hoon.'
)

payload = json.dumps({
    'model': 'fish_audio',
    'input': TEXT,
    'voice': 'prakash_confident',
    'response_format': 'pcm',
    'stream': True,
    'stream_format': 'audio'
}).encode()

conn = http.client.HTTPConnection('127.0.0.1', 8020, timeout=60)
t0   = time.perf_counter()
conn.request('POST', '/v1/audio/speech', body=payload,
             headers={'Content-Type': 'application/json', 'Accept': 'audio/pcm'})
resp = conn.getresponse()

print(f'HTTP {resp.status}  Transfer-Encoding: {resp.getheader("Transfer-Encoding")}')
assert resp.status == 200, f'Expected 200, got {resp.status}: {resp.read().decode()}'

chunks, ttfa = [], None
while True:
    c = resp.read(4096)
    if not c: break
    if ttfa is None:
        ttfa = time.perf_counter() - t0
    chunks.append(c)
conn.close()

total = sum(len(c) for c in chunks)
print(f'\nChunks     : {len(chunks)}')
print(f'Total bytes: {total:,}')
print(f'TTFA       : {ttfa*1000:.0f} ms')
print(f'Duration   : {total/(44100*2):.2f}s  (16-bit mono @44100)')

assert len(chunks) >= 2, '⚠️ Only 1 chunk — not streaming! Check mode: streaming in config'
print('\n✅ PASS — streaming confirmed!')

In [ ]:
# 11. Save & play the streamed audio
import wave, IPython.display as ipd

WAV = '/content/streaming_output.wav'
with wave.open(WAV, 'wb') as wf:
    wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(44100)
    wf.writeframes(b''.join(chunks))

print(f'Saved: {WAV}')
ipd.display(ipd.Audio(WAV))

In [ ]:
# 12. TTFA benchmark — 5 sentences × 3 runs
import http.client, json, time, statistics

SENTENCES = [
    'Namaste, main aapki kaise madad kar sakta hoon?',
    'Aapka account number kya hai?',
    'Ek minute, main check kar raha hoon.',
    'Aapke account mein transaction pending hai.',
    'Dhanyawad, aur koi sawaal ho toh zaroor puchhein.',
]

results = []
for sent in SENTENCES:
    ttfas = []
    for run in range(3):
        payload = json.dumps({
            'model': 'fish_audio', 'input': sent,
            'voice': 'prakash_confident',
            'response_format': 'pcm', 'stream': True, 'stream_format': 'audio'
        }).encode()
        conn = http.client.HTTPConnection('127.0.0.1', 8020, timeout=60)
        t0   = time.perf_counter()
        conn.request('POST', '/v1/audio/speech', body=payload,
                     headers={'Content-Type': 'application/json'})
        resp = conn.getresponse()
        ttfa, nchunks = None, 0
        while True:
            c = resp.read(4096)
            if not c: break
            if ttfa is None: ttfa = time.perf_counter() - t0
            nchunks += 1
        conn.close()
        ttfas.append(ttfa * 1000)
        print(f'  {sent[:35]:<35} run{run+1}  TTFA={ttfa*1000:>6.0f}ms  chunks={nchunks}')
    results.append({'text': sent, 'min': min(ttfas), 'avg': statistics.mean(ttfas), 'max': max(ttfas)})

print('\n' + '='*65)
print(f'{"Sentence":<38} {"Min":>6} {"Avg":>6} {"Max":>6}  (ms)')
print('='*65)
for r in results:
    print(f'{r["text"][:38]:<38} {r["min"]:>6.0f} {r["avg"]:>6.0f} {r["max"]:>6.0f}')
print('='*65)
overall = statistics.mean(r['avg'] for r in results)
print(f'Overall avg TTFA: {overall:.0f} ms')
print('✅ PASS' if overall < 2000 else '⚠️  > 2000ms — tune text_chunk_size')

In [ ]:
# 13. Run the full measure_tts_streaming_client.py benchmark
!python {REPO}/tools/streaming/measure_tts_streaming_client.py \
    --host 127.0.0.1 --port 8020 \
    --voice prakash_confident \
    --text "Namaste! Aaj hum aapke account ke baare mein baat karenge. Kya aap mujhe apna registered number bata sakte hain? Main aapki madad ke liye hoon." \
    --runs 3 --format pcm --stream-format audio

In [ ]:
# 14. Cleanup
import os, signal
try: os.killpg(os.getpgid(proc.pid), signal.SIGTERM); print('Server stopped')
except: pass
try: cf.terminate(); print('Tunnel stopped')
except: pass